In [1]:
import torch
import torch.nn as nn

In [2]:
import pandas as pd

dst = pd.read_csv("data.csv")
dst.head()

,0
0,535+278=813
1,437+985=1422
2,70+959=1029
3,763+355=1118
4,3+514=517


In [3]:
dst.dtypes

,0
0,object


In [4]:
from sklearn.model_selection import train_test_split

train_dst, test_dst = train_test_split(
    dst,
    test_size=0.2,
    random_state=42
)

In [5]:
len(train_dst)

15830

In [6]:
len(test_dst)

3958

In [7]:
train_dst

,0
17982,161+443=604
2640,154+130=284
222,390+460=850
18528,630+691=1321
4318,333+226=559
...,...
11284,767+49=816
11964,593+224=817
5390,289+728=1017
860,408+820=1228


In [8]:
symbol2id = {}
id2symbol = {}

for i in range(10):
    symbol2id[str(i)] = i
    id2symbol[i] = str(i)

EQUAL_ID = 10
symbol2id["="] = EQUAL_ID
id2symbol[EQUAL_ID] = "="

PLUS_ID = 11
symbol2id["+"] = PLUS_ID
id2symbol[PLUS_ID] = "+"

EOS_ID = 12
symbol2id["<EOS>"] = EOS_ID
id2symbol[EOS_ID] = "<EOS>"

PAD_ID = 13
symbol2id["<PAD>"] = PAD_ID
id2symbol[PAD_ID] = "<PAD>"


In [9]:
len(symbol2id)

14

In [10]:
from typing import List

In [11]:
def encode(example: str) -> List[int]:
    result = []

    for symbol in example:
        result.append(symbol2id[symbol])

    result.append(EOS_ID)

    return result

def decode(example: torch.Tensor) -> str:
    result = ""
    for symbol in example:
        result += id2symbol[symbol.item()]

    return result



In [12]:
train_dst

,0
17982,161+443=604
2640,154+130=284
222,390+460=850
18528,630+691=1321
4318,333+226=559
...,...
11284,767+49=816
11964,593+224=817
5390,289+728=1017
860,408+820=1228


In [13]:
vocab_size = 12
hidden_size = 10
emb_layer = nn.Embedding(vocab_size, hidden_size)

# emb_layer(a)

In [14]:
train_dst

,0
17982,161+443=604
2640,154+130=284
222,390+460=850
18528,630+691=1321
4318,333+226=559
...,...
11284,767+49=816
11964,593+224=817
5390,289+728=1017
860,408+820=1228


In [15]:
def collate_fn(examples: List[str]):
    encoded_examples = []
    targets = []
    max_length = 0

    for example in examples:
        if len(example) > max_length:
            max_length = len(example)

    max_length += 1

    for example in examples:
        enc_ex = encode(example)
        enc_ex.extend([PAD_ID] * (max_length - len(enc_ex)))
        encoded_examples.append(
            torch.tensor(enc_ex, dtype=torch.int))

        equal_id = enc_ex.index(EQUAL_ID)
        target = enc_ex.copy()
        target[:equal_id+1] = [PAD_ID] * (equal_id + 1)
        targets.append(
            torch.tensor(target, dtype= torch.long))


    return (torch.stack(encoded_examples, dim=0),
            torch.stack(targets, dim=0))


In [16]:
encode('773+864=1234')

[7, 7, 3, 11, 8, 6, 4, 10, 1, 2, 3, 4, 12]

In [17]:
collate_fn(['773+864=1234', "1+2=3"])

(tensor([[ 7,  7,  3, 11,  8,  6,  4, 10,  1,  2,  3,  4, 12],
         [ 1, 11,  2, 10,  3, 12, 13, 13, 13, 13, 13, 13, 13]],
        dtype=torch.int32),
 tensor([[13, 13, 13, 13, 13, 13, 13, 13,  1,  2,  3,  4, 12],
         [13, 13, 13, 13,  3, 12, 13, 13, 13, 13, 13, 13, 13]]))

In [18]:
collate_fn(['773+864=1234', "1+2=3"])[1]

tensor([[13, 13, 13, 13, 13, 13, 13, 13,  1,  2,  3,  4, 12],
        [13, 13, 13, 13,  3, 12, 13, 13, 13, 13, 13, 13, 13]])

In [19]:
vocab_size = len(symbol2id)
hidden_size = 10

emb_layer = nn.Embedding(vocab_size, hidden_size)


In [20]:
emb_layer(collate_fn(['773+864=1234', "1+2=3"])[1]).shape

torch.Size([2, 13, 10])

In [21]:
from torch.utils.data import Dataset

class DatasetTrain(Dataset):
    def __init__(self, data: List[str]):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


In [22]:
trainDataSet = DatasetTrain(train_dst.iloc[:,0].tolist())

In [23]:
testDataSet = DatasetTrain(test_dst.iloc[:,0].tolist())

In [24]:
trainDataSet[0]

'161+443=604'

In [25]:
from torch.utils.data import DataLoader

trainDataLoader = DataLoader(
    dataset=trainDataSet,
    batch_size=5,
    collate_fn=collate_fn,
    shuffle=True
)

In [26]:
k = 0
for addition, targets in trainDataLoader:
    if k == 0:
        i = addition
        print("Addition")
        print(i)
        print(type(i))
        print(i.shape)

        i = targets
        print("Targets")
        print(i)
        print(type(i))
        print(i.shape)

        k += 1


Addition
tensor([[ 6,  2,  6, 11,  3,  5,  6, 10,  9,  8,  2, 12, 13],
        [ 8,  9,  0, 11,  6,  9,  0, 10,  1,  5,  8,  0, 12],
        [ 7,  3,  6, 11,  1,  9,  0, 10,  9,  2,  6, 12, 13],
        [ 4,  4,  9, 11,  2,  7,  4, 10,  7,  2,  3, 12, 13],
        [ 6,  1,  5, 11,  5,  8,  1, 10,  1,  1,  9,  6, 12]],
       dtype=torch.int32)
<class 'torch.Tensor'>
torch.Size([5, 13])
Targets
tensor([[13, 13, 13, 13, 13, 13, 13, 13,  9,  8,  2, 12, 13],
        [13, 13, 13, 13, 13, 13, 13, 13,  1,  5,  8,  0, 12],
        [13, 13, 13, 13, 13, 13, 13, 13,  9,  2,  6, 12, 13],
        [13, 13, 13, 13, 13, 13, 13, 13,  7,  2,  3, 12, 13],
        [13, 13, 13, 13, 13, 13, 13, 13,  1,  1,  9,  6, 12]])
<class 'torch.Tensor'>
torch.Size([5, 13])


In [27]:
import numpy as np

tensor = np.arange(12).reshape(3, 2, 2)
print(tensor)


[[[ 0  1]
  [ 2  3]]

 [[ 4  5]
  [ 6  7]]

 [[ 8  9]
  [10 11]]]


In [28]:
print(tensor[:,:,0])

[[ 0  2]
 [ 4  6]
 [ 8 10]]


In [29]:
class RNNAddition(nn.Module):

    def __init__(self, vocab_size:int, hidden_size:int):
        super().__init__()
        self.emb_layer = nn.Embedding(vocab_size, hidden_size)
        self.hidden_size = hidden_size
        self.W = nn.Linear(2*hidden_size,hidden_size)
        self.O = nn.Linear(hidden_size, vocab_size)

    def forward(self, inputs, h_0 = None):
        batch_size, seq_len = inputs.shape
        outputs = []
        inputs = self.emb_layer(inputs)

        if h_0 is None:
            h_t = torch.zeros(batch_size, self.hidden_size, device = inputs.device)
        else:
            h_t = h_0

        for i in range(seq_len):
            x_t = inputs[:,i,:]
            h_t = torch.tanh(self.W(torch.cat([x_t, h_t], dim = -1)))
            output = self.O(h_t)
            outputs.append(output)

        return h_t, torch.stack(outputs, dim= 1)


In [30]:
print(len(symbol2id))

14


In [31]:
@torch.no_grad()

def evaluation(model, dataloader, loss_fn):
  model.eval()

  losses = []
  accuracies = []
  exact_matches = []

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  for inputs_id, targets in dataloader:
      inputs_id = inputs_id.to(device)
      targets = targets.to(device)

      _, outputs = model(inputs_id)
      outputs = outputs[:,:-1]
      targets = targets[:, 1:]

      loss = loss_fn(outputs.permute(0, 2, 1), targets)

      outputs = outputs.argmax(-1)
      mask = (targets != PAD_ID)
      corrects = (outputs == targets) & mask
      accuracy = corrects.sum().item() / mask.sum().item()

      accuracies.append(accuracy)
      losses.append(loss.item())

        #exact match
      pad_mask = (targets == PAD_ID)
      token_ok = (outputs == targets) | pad_mask
      exact_per_sample = token_ok.all(dim=1)
      exact_match = exact_per_sample.float().mean().item()
      exact_matches.append(exact_match)


  loss = np.mean(losses)
  accuracy = np.mean(accuracies)
  exact_match = np.mean(exact_matches)

  return loss, accuracy, exact_match


In [32]:
from tqdm import tqdm

In [35]:
def train(model, dataloader, loss_fn, optimizer):

    model.train()

    losses = []
    accuracies = []
    exact_matches = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for inputs_id, targets in dataloader:
        inputs_id = inputs_id.to(device)
        targets = targets.to(device)

        _, outputs = model(inputs_id)
        outputs = outputs[:,:-1]
        targets = targets[:, 1:]

        loss = loss_fn(outputs.permute(0, 2, 1), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        outputs = outputs.argmax(-1)
        mask = (targets != PAD_ID)
        corrects = (outputs == targets) & mask
        accuracy = corrects.sum().item() / mask.sum().item()

        #accuracies.append((outputs.argmax(-1) == targets).float().mean().item())
        #accuracy
        accuracies.append(accuracy)
        losses.append(loss.item())

        #exact match
        pad_mask = (targets == PAD_ID)
        token_ok = (outputs == targets) | pad_mask
        exact_per_sample = token_ok.all(dim=1)
        exact_match = exact_per_sample.float().mean().item()
        exact_matches.append(exact_match)


    loss = np.mean(losses)
    accuracy = np.mean(accuracies)
    exact_match = np.mean(exact_matches)

    return loss, accuracy, exact_match

In [40]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

def train_rnn_model(model, num_epochs: int = 200, patience: int = 10, min_delta: float = 1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    print('Number of parameters:', sum(p.numel() for p in model.parameters()))

    optimizer = torch.optim.Adam(model.parameters(), lr=4e-4, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    train_loader = DataLoader(trainDataSet, batch_size=128, collate_fn=collate_fn, shuffle=True)
    test_loader  = DataLoader(testDataSet,  batch_size=128, collate_fn=collate_fn, shuffle=False)

    best_val_loss = np.inf
    bad_epochs = 0

    for epoch in tqdm(range(1, num_epochs + 1)):
        train_loss, train_acc, train_em = train(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc, val_em = evaluation(model, test_loader, loss_fn)

        if epoch % 50 == 0 or epoch == 1:
            print()
            print(f"[{epoch}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_em={train_em:.4f}")
            print(f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_em={val_em:.4f}")

        improved = (best_val_loss - val_loss) > min_delta
        if improved:
            best_val_loss = val_loss
            bad_epochs = 0
            torch.save(model.state_dict(), "best_rnn.pt")
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping: val_loss does not improve {patience} epochs. "
                  f"Best val_loss={best_val_loss:.4f}")
            break

    model.load_state_dict(torch.load("best_rnn.pt", map_location=device))
    return model

In [41]:
rrr = RNNAddition(len(symbol2id), 256)
train_rnn_model(rrr, 350)

Number of parameters: 138510


  0%|          | 1/350 [00:01<06:45,  1.16s/it]


[1] train_loss=1.7373 train_acc=0.3859 train_em=0.0009
val_loss=1.5468 val_acc=0.4179 val_em=0.0015


 14%|█▍        | 50/350 [01:02<06:28,  1.29s/it]


[50] train_loss=0.9353 train_acc=0.6526 train_em=0.0429
val_loss=0.9453 val_acc=0.6401 val_em=0.0311


...

 71%|███████▏  | 250/350 [05:10<02:17,  1.37s/it]


[250] train_loss=0.0826 train_acc=0.9806 train_em=0.9251
val_loss=0.1073 val_acc=0.9659 val_em=0.8741


 74%|███████▎  | 258/350 [05:23<01:55,  1.25s/it]

Early stopping: val_loss does not improve 10 epochs. Best val_loss=0.1002


RNNAddition(
  (emb_layer): Embedding(14, 256)
  (W): Linear(in_features=512, out_features=256, bias=True)
  (O): Linear(in_features=256, out_features=14, bias=True)
)

In [42]:
def strange_encode(example: str) -> List[int]:
    result = []

    for symbol in example:
        result.append(symbol2id[symbol])
    return result

In [50]:
def generate_answer(model, example: str, max_new_tokens: int = 10):
    model.eval()
    device = next(model.parameters()).device

    prefix = torch.tensor(strange_encode(example), dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        h_t, out = model(prefix)  # out: [1, seq_len, vocab]
        next_id = out[:, -1, :].argmax(-1).item() \

    generated = []
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if next_id == EOS_ID:
                break
            generated.append(id2symbol[next_id])

            inp = torch.tensor([[next_id]], dtype=torch.long, device=device)
            h_t, out = model(inp, h_0=h_t)
            next_id = out[:, -1, :].argmax(-1).item()

    return "".join(generated)

In [62]:
generate_answer(rrr, '697+124=')

'821'

In [57]:
dst_4d = pd.read_csv("data_4_19997.csv")
dst_4d.head()

,0
0,8510+8322=16832
1,4441+9516=13957
2,2147+1034=3181
3,4175+9425=13600
4,9320+5031=14351


In [58]:
testDataSet4d = DatasetTrain(dst_4d.iloc[:,0].tolist())

In [63]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
test_loader  = DataLoader(testDataSet,  batch_size=128, collate_fn=collate_fn, shuffle=False)
evaluation(rrr, test_loader, loss_fn)

(np.float64(0.10022100481775499),
 np.float64(0.9709243054192991),
 np.float64(0.8936022077837298))

In [59]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
test_loader_4d  = DataLoader(testDataSet4d,  batch_size=128, collate_fn=collate_fn, shuffle=False)
evaluation(rrr, test_loader_4d, loss_fn)

(np.float64(7.575279782532127),
 np.float64(0.26344714902144234),
 np.float64(0.0))